# Data Decoding: Canadian Community Health Survey (CCHS) 2019-2020 Public Use Microdata File  

## Overview  
This document outlines the process of decoding the original Canadian Community Health Survey (CCHS) 2019-2020 data using the provided dictionary documents.  

## CCHS Content Structure  
The CCHS consists of two main components:  
1. **Core Content** – Asked to all respondents.  
2. **Optional Content** – Selected by provincial and territorial stakeholders in coordination with health regions and only asked in the provinces and territories that chose the module.  

## Feature Selection Criteria  
Out of approximately 690 available features, only those relevant to our project were selected. The key considerations for feature selection included:  
- **Core Content Inclusion** – Features included in the survey for all respondents were prioritized.  
- **Irrelevant Features Excluded** – Features such as sequential record numbers, date of file creation, reference periods, etc., were removed.  
- **Short-Term Data Excluded** – Data collected for a very short duration, such as "Drank alcohol – past week," "Number of drinks – Day 1, Day 2 ... Day 7 of the past week," etc., were excluded.  

## Final Selection  
After applying the above considerations, **32 features** were selected for our study.  

### Note:  
Extensively documentations are reviewed and analyzed to ensure appropriate feature selection and accurate decoding.


In [9]:
import os
import pandas as pd

# --- Path Config ---
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
STATS_DIR = os.path.join(OUTPUTS_DIR, "statistics")
PLOTS_DIR = os.path.join(OUTPUTS_DIR, "plots")
METRICS_DIR = os.path.join(OUTPUTS_DIR, "metrics")

# Make sure subfolders exist
os.makedirs(STATS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)


In [10]:
import pandas as pd

# Loading the dataset
df = pd.read_csv(os.path.join(DATA_DIR, "pumf_cchs.csv"))

# Basic checks
print(df.head())  
print(df.info())  

   ADM_RNO1   VERDATE     REFPER  GEOGPRV  GEODGHR4  DHH_SEX  DHHGMS  \
0      1000  20240531  2019-2020     47.0   47906.0      2.0     1.0   
1    100005  20240531  2019-2020     47.0   47906.0      1.0     1.0   
2    100012  20240531  2019-2020     59.0   59914.0      2.0     2.0   
3    100015  20240531  2019-2020     13.0   13904.0      1.0     2.0   
4    100018  20240531  2019-2020     46.0   46903.0      1.0     2.0   

   DHHDGHSZ  ADM_PRX  DHHGAGE  ...  FSCDVHF2  INCG015  INCDGHH  INCDGRCA  \
0       2.0      2.0      3.0  ...       0.0      1.0      5.0       4.0   
1       2.0      2.0      5.0  ...       0.0      2.0      4.0       2.0   
2       1.0      2.0      5.0  ...       6.0      2.0      2.0       1.0   
3       1.0      2.0      5.0  ...       0.0      2.0      3.0       3.0   
4       1.0      2.0      4.0  ...       0.0      2.0      1.0       1.0   

   INCDGRPR  INCDGRRS  ADM_040  ADM_045  ADM_050   WTS_M  
0       4.0       4.0      1.0      1.0      6.0  7

In [11]:

selected_columns = {
    # Demographic
    "DHHGAGE": "Age Group",
    "DHH_SEX": "Sex at Birth",
    "DHHGMS": "Marital Status",

    # Sucide
    "SUI_005": "Considered suicide - lifetime",
    "SUI_010": "Considered suicide - last 12 months",

    # Smoking
    "SMKDVSTY": "Smoking status",

    # Canabis
    "SDSDVTOT": "Severity of Canabis Dependence",
    "CAN_015": "Used cannabis - 12 months",

    # Primary Health care
    "PHC_005": "Usual place for immediate care for minor problem",
    
    # Income 
    "INCDGHH": "Total Household Income - All Sources",

    # BMI
    "HWTDGWHO": "BMI age 12 to 17 (self-reported) - WHO classification",
    "HWTDGBCC": "BMI classification for adults aged 18 and over (adjusted) - international",

    # Health Utility
    "HUIDGPAD": "Pain health status",

    # General Health
    "GENDVHDI": "Perceived health ",
    "GENDVMHI": "Perceived mental health ",
    "GENDVSWL": "Satisfaction with life in general ",

    # For FLU 
    "FLU_005": "Had a seasonal flu shot (excluding H1N1) - lifetime",
    "FLU_010": "Seasonal flu shot - last time",

    # Alcohol
    "ALCDVTTM": "Type of drinker",
    "ALC_020": "Drank 5+ / 4+ drinks one occasion - frequency - 12 months",

    # Chronic Conditions
    "CCC_035": "Has sleep apnea",
    "CCC_075": "Has high blood cholesterol / lipids",
    "CCC_080": "High blood cholesterol / lipids - took medication - 1 month",
    "CCC_185": "Has chronic fatigue syndrome",
    "CCC_195": "Has a mood disorder (depression, bipolar, mania, dysthymia)",
    "CCC_200": "Has an anxiety disorder (phobia, OCD, panic)",
    "CCCDGRSP": "Has respiratory chronic condition (asthma or COPD)",
    "CCCDGSKL": "Musculoskeletal condition (Arthritis, fibromyalgia, osteoporosis)",

    # Chronic and Target
    "CCC_070": "High blood pressure - took medication - 1 month", # not target but associated with High blood pressure
    "CCC_065": "Has a high blood pressure",
    "CCC_095": "Has diabetes",
    "CCCDGCAR": "Cardiovascular condition (Heart disease or stroke)"
    

}


value_mappings = {
    #Demographic
    "DHHGAGE": {1: "12-17 years", 2: "18 to 34 years", 3: "35 to 49 years", 4: "50 to 64 years", 5: "65 and older"},
    "DHH_SEX": {1: "Male", 2: "Female"},
    "DHHGMS": {1: "Married/Common-law", 2: "Widowed/Divorced/Separated/Single, never married", 6: "Valid skip (under 18)", 9: "Not stated"},

    # Sucide
    "SUI_005": {1: "Yes", 2: "No", 6: "Valid skip", 7: "Don’t know", 8: "Refusal", 9: "Not stated"},
    "SUI_010": {1: "Yes", 2: "No", 6: "Valid skip", 7: "Don’t know", 8: "Refusal", 9: "Not stated"},

    # Smoking
    "SMKDVSTY": {1: "Current daily smoker", 2: "Current occasional smoker", 
                  3: "Former daily smoker (non-smoker now)", 4: "Former occasional smoker (non-smoker now)", 
                  5: "Experimental smoker (at least 1 cig, non-smoker now)", 6: "Lifetime abstainer (never smoked)", 
                  99: "Not stated"},

    # Canabis 
    "SDSDVTOT": {0: "0", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5", 6: "6", 7: "7", 8: "8", 9: "9", 10: "10", 11: "11", 
                 12: "12", 13: "13", 14: "14", 15: "15", 96: "Valid skip", 99: "Not stated"},
    "CAN_015": {1: "Yes", 2: "No", 7: "Don’t know", 8: "Refusal", 9: "Not stated"},

    # Primary Health care
    "PHC_005": {1: "Yes", 2: "No", 7: "Don’t know", 8: "Refusal"},

    # Income 
    "INCDGHH": {1: "No income or less than $20,000", 2: "$20,000 to $39,999", 
                3: "$40,000 to $59,999", 4: "$60,000 to $79,999", 
                5: "$80,000 or more", 9: "Not stated"},

    # BMI
    "HWTDGWHO": {1: "Thinness/Normal", 2: "Overweight/Obese", 6: "Valid skip", 9: "Not stated"},
    "HWTDGBCC": {1: "Underweight/ Normal weight", 2: "Overweight / Obese - Class I, II, III", 6: "Valid skip", 9: "Not stated"},

    # Health Utility
    "HUIDGPAD": {1: "No usual pain or discomfort", 2: "Has usual pain or discomfort", 9: "Not stated"},

    # General Health
    "GENDVHDI": {0: "Poor", 1: "Fair", 2: "Good", 3: "Very good", 4: "Excellent", 9: "Not stated"},
    "GENDVMHI": {0: "Poor", 1: "Fair", 2: "Good", 3: "Very good", 4: "Excellent", 9: "Not stated"},
    "GENDVSWL": {1: "Very Satisfied", 2: "Satisfied", 3: "Neither satisfied nor dissatisfied", 
                 4: "Dissatisfied", 5: "Very Dissatisfied", 9: "Not stated"},

    # For FLU
    "FLU_005": {1: "Yes", 2: "No", 7: "Don’t know", 8: "Refusal", 9: "Not stated"},
    "FLU_010": {1: "Less than 1 year ago", 2: "1 year to less than 2 years ago", 3: "2 years ago or more", 
                6: "Valid skip", 7: "Don’t know", 8: "Refusal", 9: "Not stated"},

    # Alcohol
    "ALCDVTTM": {1: "Regular drinker", 2: "Occasional drinker", 3: "Did not drink in the last 12 months", 9: "Not stated"}, 
    "ALC_020": {1: "Never", 2: "Less than once a month", 3: "Once a month", 
                4: "2-3 times a month", 5: "Once a week", 6: "More than once a week", 
                96: "Valid skip", 97: "Don’t know", 98: "Refusal", 99: "Not stated"},

    # Chronic Conditions
    "CCC_035": {1: "Yes", 2: "No", 7: "Don’t know", 8: "Refusal"},
    "CCC_075": {1: "Yes", 2: "No", 6: "Valid skip", 7: "Don’t know", 8: "Refusal"},
    "CCC_080": {1: "Yes", 2: "No", 6: "Valid skip", 7: "Don’t know", 8: "Refusal"},
    "CCC_185": {1: "Yes", 2: "No", 7: "Don’t know", 8: "Refusal"},
    "CCC_195": {1: "Yes", 2: "No", 7: "Don’t know", 8: "Refusal"},
    "CCC_200": {1: "Yes", 2: "No", 7: "Don’t know", 8: "Refusal"},
    "CCCDGRSP": {1: "Yes", 2: "No", 9: "Not stated"},
    "CCCDGSKL": {1: "Yes", 2: "No", 6: "Valid skip", 9: "Not stated"},

    # Chronic and Target
    "CCC_070": {1: "Yes", 2: "No", 7: "Don’t know", 8: "Refusal"},
    "CCC_065": {1: "Yes", 2: "No", 7: "Don’t know", 8: "Refusal"},
    "CCC_095": {1: "Yes", 2: "No", 7: "Don’t know", 8: "Refusal", 9: "Not stated"},
    "CCCDGCAR": {1: "Yes", 2: "No", 9: "Not stated"},
    
}

# Check if all selected columns are present in the dataset
missing_cols = [col for col in selected_columns.keys() if col not in df.columns]
if missing_cols:
    print("Warning: The following columns are missing from the dataset:", missing_cols)


# Subset the dataframe with selected columns
# Select all rows from the dataset for selected columns
df_subset = df[list(selected_columns.keys())]

# Check if the subset contains all selected columns
print("Columns in df_subset:", df_subset.columns.tolist())
print("Expected columns:", list(selected_columns.keys()))

# Rename columns for better readability
df_subset_original = df_subset.rename(columns=selected_columns)

# Check if the columns are renamed correctly
print("Renamed columns in df_subset_original:", df_subset_original.columns.tolist())
print("Expected renamed columns:", list(selected_columns.values()))


# Apply value mappings to decode all categorical values
df_subset_decoded = df_subset.copy()
for col, mapping in value_mappings.items():
    if col in df_subset_decoded.columns:
        print(f"\nColumn: {col}")
        print("Unique values before mapping:", df_subset[col].unique())
        df_subset_decoded[col] = df_subset_decoded[col].map(mapping)
        print("Unique values after mapping:", df_subset_decoded[col].unique())

# Rename columns for better readability
df_subset_decoded = df_subset_decoded.rename(columns=selected_columns)


Columns in df_subset: ['DHHGAGE', 'DHH_SEX', 'DHHGMS', 'SUI_005', 'SUI_010', 'SMKDVSTY', 'SDSDVTOT', 'CAN_015', 'PHC_005', 'INCDGHH', 'HWTDGWHO', 'HWTDGBCC', 'HUIDGPAD', 'GENDVHDI', 'GENDVMHI', 'GENDVSWL', 'FLU_005', 'FLU_010', 'ALCDVTTM', 'ALC_020', 'CCC_035', 'CCC_075', 'CCC_080', 'CCC_185', 'CCC_195', 'CCC_200', 'CCCDGRSP', 'CCCDGSKL', 'CCC_070', 'CCC_065', 'CCC_095', 'CCCDGCAR']
Expected columns: ['DHHGAGE', 'DHH_SEX', 'DHHGMS', 'SUI_005', 'SUI_010', 'SMKDVSTY', 'SDSDVTOT', 'CAN_015', 'PHC_005', 'INCDGHH', 'HWTDGWHO', 'HWTDGBCC', 'HUIDGPAD', 'GENDVHDI', 'GENDVMHI', 'GENDVSWL', 'FLU_005', 'FLU_010', 'ALCDVTTM', 'ALC_020', 'CCC_035', 'CCC_075', 'CCC_080', 'CCC_185', 'CCC_195', 'CCC_200', 'CCCDGRSP', 'CCCDGSKL', 'CCC_070', 'CCC_065', 'CCC_095', 'CCCDGCAR']
Renamed columns in df_subset_original: ['Age Group', 'Sex at Birth', 'Marital Status', 'Considered suicide - lifetime', 'Considered suicide - last 12 months', 'Smoking status', 'Severity of Canabis Dependence', 'Used cannabis - 12 m

In [12]:
# Check for NaN values after mapping
nan_counts = df_subset_decoded.isna().sum()
if nan_counts.sum() > 0:
    print("Columns with NaN values after mapping:")
    print(nan_counts[nan_counts > 0])

# Check if the columns are renamed correctly
print("Final columns in df_subset_decoded:", df_subset_decoded.columns.tolist())
print("Expected renamed columns:", list(selected_columns.values()))

Final columns in df_subset_decoded: ['Age Group', 'Sex at Birth', 'Marital Status', 'Considered suicide - lifetime', 'Considered suicide - last 12 months', 'Smoking status', 'Severity of Canabis Dependence', 'Used cannabis - 12 months', 'Usual place for immediate care for minor problem', 'Total Household Income - All Sources', 'BMI age 12 to 17 (self-reported) - WHO classification', 'BMI classification for adults aged 18 and over (adjusted) - international', 'Pain health status', 'Perceived health ', 'Perceived mental health ', 'Satisfaction with life in general ', 'Had a seasonal flu shot (excluding H1N1) - lifetime', 'Seasonal flu shot - last time', 'Type of drinker', 'Drank 5+ / 4+ drinks one occasion - frequency - 12 months', 'Has sleep apnea', 'Has high blood cholesterol / lipids', 'High blood cholesterol / lipids - took medication - 1 month', 'Has chronic fatigue syndrome', 'Has a mood disorder (depression, bipolar, mania, dysthymia)', 'Has an anxiety disorder (phobia, OCD, panic

In [13]:
# Export the encoded data
df_subset_original.to_csv(os.path.join(STATS_DIR, "selected_encoded_data.csv"), index=False)

# Export the fully decoded data
df_subset_decoded.to_csv(os.path.join(STATS_DIR, "selected_decoded_data.csv"), index=False)

print("Files 'selected_encoded_data.csv' and 'selected_decoded_data.csv' have been saved successfully.")



Files 'selected_encoded_data.csv' and 'selected_decoded_data.csv' have been saved successfully.


# ABT (Analytic Base Table)

| **S. No.** | **Feature Name**                                                                 | **Domain Concept**             | **Feature Description**                                                                 | **Feature Type** | **Data Type** |
|------------|----------------------------------------------------------------------------------|-------------------------------|------------------------------------------------------------------------------------------|------------------|---------------|
| 1          | Age Group                                                                        | Demographics                  | Age category of the respondent (e.g., 18–34, 35–49, etc.).                              | Categorical      | String        |
| 2          | Sex at Birth                                                                     | Demographics                  | Sex assigned at birth: Male or Female.                                                  | Categorical      | String        |
| 3          | Marital Status                                                                   | Demographics                  | Marital status of the respondent.                                                       | Categorical      | String        |
| 4          | Considered suicide - lifetime                                                    | Mental Health                 | Whether respondent has ever considered suicide in their lifetime.                      | Categorical      | String        |
| 5          | Considered suicide - last 12 months                                              | Mental Health                 | Whether respondent considered suicide in the last 12 months.                            | Categorical      | String        |
| 6          | Smoking status                                                                   | Lifestyle                     | Smoking behavior including current, former, or never smoked.                            | Categorical      | String        |
| 7          | Severity of Canabis Dependence                                                   | Substance Use                 | Level of cannabis dependence (if applicable).                                            | Categorical      | String        |
| 8          | Used cannabis - 12 months                                                        | Substance Use                 | Whether cannabis was used in the past 12 months.                                        | Categorical      | String        |
| 9          | Usual place for immediate care for minor problem                                 | Access to Care                | Whether respondent has a usual place for minor health problems.                         | Categorical      | String        |
| 10         | Total Household Income - All Sources                                             | Socioeconomic                 | Total annual household income from all sources.                                         | Categorical      | String        |
| 11         | BMI age 12 to 17 (self-reported) - WHO classification                            | Health Metrics                | BMI classification for ages 12–17 (self-reported, WHO standards).                      | Categorical      | String        |
| 12         | BMI classification for adults aged 18 and over (adjusted) - international        | Health Metrics                | BMI classification for adults (based on international standards).                       | Categorical      | String        |
| 13         | Pain health status                                                               | Health Status                 | Indicates whether the respondent has usual pain or discomfort.                          | Categorical      | String        |
| 14         | Perceived health - (D)                                                           | Health Perception             | Self-rated overall physical health.                                                     | Categorical      | String        |
| 15         | Perceived mental health - (D)                                                    | Mental Health                 | Self-rated overall mental health.                                                       | Categorical      | String        |
| 16         | Satisfaction with life in general - (D)                                          | Well-being                    | Self-reported satisfaction with life in general.                                        | Categorical      | String        |
| 17         | Had a seasonal flu shot (excluding H1N1) - lifetime                              | Immunization History          | Whether respondent has ever had a seasonal flu shot.                                    | Categorical      | String        |
| 18         | Seasonal flu shot - last time                                                    | Immunization History          | When the last seasonal flu shot was taken.                                              | Categorical      | String        |
| 19         | Type of drinker                                                                  | Lifestyle                     | Drinking behavior classification (e.g., regular, occasional, abstainer).                | Categorical      | String        |
| 20         | Drank 5+ / 4+ drinks one occasion - frequency - 12 months                        | Lifestyle                     | Frequency of binge drinking over the past 12 months.                                    | Categorical      | String        |
| 21         | Has sleep apnea                                                                  | Diagnosed Conditions          | Whether the respondent has been diagnosed with sleep apnea.                             | Categorical      | String        |
| 22         | Has high blood cholesterol / lipids                                              | Diagnosed Conditions          | Indicates if respondent has high cholesterol or lipids.                                 | Categorical      | String        |
| 23         | High blood cholesterol / lipids - took medication - 1 month                      | Medication Use                | Whether medication was taken for high cholesterol in the past month.                    | Categorical      | String        |
| 24         | Has chronic fatigue syndrome                                                     | Diagnosed Conditions          | Whether respondent has chronic fatigue syndrome.                                        | Categorical      | String        |
| 25         | Has a mood disorder (depression, bipolar, mania, dysthymia)                      | Mental Health                 | Whether respondent has a diagnosed mood disorder.                                       | Categorical      | String        |
| 26         | Has an anxiety disorder (phobia, OCD, panic)                                     | Mental Health                 | Whether respondent has a diagnosed anxiety disorder.                                    | Categorical      | String        |
| 27         | Has respiratory chronic condition (asthma or COPD)                               | Diagnosed Conditions          | Indicates presence of chronic respiratory illness.                                      | Categorical      | String        |
| 28         | Musculoskeletal condition (Arthritis, fibromyalgia, osteoporosis)                | Diagnosed Conditions          | Indicates presence of any musculoskeletal condition.                                    | Categorical      | String        |
| 29         | High blood pressure - took medication - 1 month                                  | Medication Use                | Whether respondent took medication for high BP in the past month.                       | Categorical      | String        |
| 30         | Has a high blood pressure                                                        | Diagnosed Conditions          | Indicates presence of high blood pressure.                                              | Categorical/Target      | String        |
| 31         | Has diabetes                                                                     | Diagnosed Conditions          | Indicates if respondent has diabetes.                                                   | Categorical/Target      | String        |
| 32         | Cardiovascular condition (Heart disease or stroke)                               | Diagnosed Conditions          | Indicates presence of any cardiovascular condition such as heart disease or stroke.     | Categorical/Target      | String        |
